# NB03 — Extracción masiva de features

## Propósito

Aplicar el pipeline validado en NB02 sobre los **5.000 estudios** de VinDr-Mammo y guardar los vectores pooleados en disco. Tiempo estimado: ~45 minutos en GPU.

## Estrategia de almacenamiento

**Guardamos los vectores pooleados por vista y por par bilateral**, no los tensores enteros. Razón: con los pooleados (1024 dims por unidad) podemos reconstruir cualquier configuración (E-A, E-B, M-A) por simple concatenación, sin volver a procesar imágenes.

Archivos resultantes:

| Archivo | Shape | Contenido | Tamaño |
|---------|-------|-----------|--------|
| `X_view.npy` | `(N, 4, 1024)` | GAP+GMP de cada vista, orden: L-CC, R-CC, L-MLO, R-MLO | ~80 MB |
| `X_asym.npy` | `(N, 2, 1024)` | GAP+GMP de la asimetría bilateral, orden: CC, MLO | ~40 MB |
| `metadata.csv` | N filas | study_id, split, BI-RADS, densidad, etiquetas binarias | <1 MB |

Total: ~120 MB. Manejable.

## Guardado incremental y reanudación

Si la extracción se interrumpe (caída de luz, cancelación manual, error puntual), no queremos perder lo ya procesado. Por eso:

1. Cada 100 estudios procesados se vuelca todo a disco (X_view, X_asym, metadata, lista de hechos).
2. Al arrancar, se lee `done_studies.txt` y los buffers parciales, y se salta lo ya hecho.

Esto permite **parar y continuar sin perder tiempo**.

In [1]:
import os, sys, time, gc
import numpy as np
import pandas as pd
import torch
import cv2
import pydicom
from pathlib import Path

# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))
ASYMMIRAI    = os.path.join(BASE, 'AsymMirai')
DATA         = os.path.join(BASE, 'Data', 'vindr-mammo')
OUTPUTS      = os.path.join(BASE, 'outputs')
WEIGHTS      = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')
IMG_DIR      = os.path.join(DATA, 'images')
BREAST_CSV   = os.path.join(DATA, 'breast-level_annotations.csv')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, ASYMMIRAI)
sys.path.insert(0, os.path.join(ASYMMIRAI, 'asymmetry_model'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Salida: {FEATURES_DIR}')

Device: cuda
Salida: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Features


## 1. Carga del modelo (congelado) y stretch params

Idéntico al NB02. Los necesitamos para el cálculo del Punto B.

In [2]:
model = torch.load(WEIGHTS, map_location=DEVICE, weights_only=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False

USE_STRETCH = getattr(model, 'use_stretch', False)
cc_stretch  = model.cc_stretch_params.detach() if USE_STRETCH else None
mlo_stretch = model.mlo_stretch_params.detach() if USE_STRETCH else None
print(f'use_stretch = {USE_STRETCH}    alignment_space = {getattr(model,"alignment_space",None)}')

c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'mirai_localized_dif_head.LocalizedDifModel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)


use_stretch = True    alignment_space = None


## 2. Funciones de preprocesado y extracción

Mismo código que en NB02, encapsulado en una función `extract_study()` que devuelve los 6 vectores pooleados (4 vistas + 2 asimetrías) por estudio.

In [3]:
MIRAI_MEAN = 7699.5
MIRAI_STD  = 11765.06
TARGET_H, TARGET_W = 1664, 2048

def load_dicom(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
        wc = float(ds.WindowCenter[0]) if hasattr(ds.WindowCenter, '__iter__') else float(ds.WindowCenter)
        ww = float(ds.WindowWidth[0])  if hasattr(ds.WindowWidth,  '__iter__') else float(ds.WindowWidth)
        pixels = np.clip(pixels, wc-ww/2, wc+ww/2)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        pixels = pixels.max() - pixels
    return pixels

def align_by_centroid(img):
    h, w = img.shape
    mask = (img > img.mean()).astype(np.uint8)
    M = cv2.moments(mask)
    if M['m00'] == 0:
        return img
    cy, cx = int(M['m01']/M['m00']), int(M['m10']/M['m00'])
    M_aff = np.float32([[1, 0, w//2-cx], [0, 1, h//2-cy]])
    return cv2.warpAffine(img, M_aff, (w, h), borderValue=0)

def preprocess_view(dcm_path):
    img = load_dicom(dcm_path)
    img = align_by_centroid(img)
    img = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
    img = (img - MIRAI_MEAN) / MIRAI_STD
    img = np.stack([img, img, img], axis=0)[None, ...]
    return torch.from_numpy(img).float()

def pool_features(x):
    """GAP + GMP por canal: (B, C, H, W) → (B, 2C)."""
    return torch.cat([x.mean(dim=(-2, -1)), x.amax(dim=(-2, -1))], dim=1)

def compute_asym(L, R, stretch_p=None):
    if stretch_p is not None:
        sp = stretch_p.view(1, -1, 1, 1).to(L.device)
        L = sp * L; R = sp * R
    return torch.abs(L - R)

def extract_study(vistas):
    """
    vistas: dict {('L','CC'): path, ('R','CC'): path, ('L','MLO'): path, ('R','MLO'): path}
    Devuelve:
      X_view: (4, 1024) en orden [L-CC, R-CC, L-MLO, R-MLO]
      X_asym: (2, 1024) en orden [asym-CC, asym-MLO]
      o None si fallo en preprocesado.
    """
    try:
        l_cc  = preprocess_view(vistas[('L','CC')]).to(DEVICE)
        r_cc  = preprocess_view(vistas[('R','CC')]).to(DEVICE)
        l_mlo = preprocess_view(vistas[('L','MLO')]).to(DEVICE)
        r_mlo = preprocess_view(vistas[('R','MLO')]).to(DEVICE)
    except Exception as e:
        print(f'    [err preprocesado] {e}')
        return None

    with torch.no_grad():
        emb_l_cc  = model.backbone(l_cc)
        emb_r_cc  = model.backbone(r_cc)
        emb_l_mlo = model.backbone(l_mlo)
        emb_r_mlo = model.backbone(r_mlo)

        feat_lcc  = pool_features(emb_l_cc)[0]
        feat_rcc  = pool_features(emb_r_cc)[0]
        feat_lmlo = pool_features(emb_l_mlo)[0]
        feat_rmlo = pool_features(emb_r_mlo)[0]

        asym_cc  = compute_asym(emb_l_cc,  emb_r_cc,  cc_stretch  if USE_STRETCH else None)
        asym_mlo = compute_asym(emb_l_mlo, emb_r_mlo, mlo_stretch if USE_STRETCH else None)
        feat_acc  = pool_features(asym_cc)[0]
        feat_amlo = pool_features(asym_mlo)[0]

    X_view = torch.stack([feat_lcc, feat_rcc, feat_lmlo, feat_rmlo], dim=0).cpu().numpy()
    X_asym = torch.stack([feat_acc, feat_amlo], dim=0).cpu().numpy()
    return X_view, X_asym

print('Funciones definidas.')

Funciones definidas.


## 3. Construir todas las etiquetas que necesitamos

Por cada estudio guardamos varias columnas en el metadata, porque algunas las usamos ya y otras las necesitaremos en el NB07 (fusión con densidad):

- **`max_birads`**: max de las 4 vistas (define `y_estudio`).
- **`birads_L`, `birads_R`**: max BI-RADS por mama (define `y_L` y `y_R`).
- **`density`**: densidad mamaria predominante del estudio (moda).
- **`density_L`, `density_R`**: densidad por mama (para nivel mama).
- **`y_estudio`, `y_L`, `y_R`**: etiquetas binarias `BI-RADS ≥ 4`.

In [4]:
df = pd.read_csv(BREAST_CSV)

def parse_birads(v):
    if isinstance(v, str):
        s = v.replace('BI-RADS', '').replace('BIRADS', '').strip()
        try: return int(s)
        except: return np.nan
    return int(v) if not pd.isna(v) else np.nan

def parse_density(v):
    if isinstance(v, str):
        return v.replace('DENSITY', '').strip()
    return v

df['birads_int'] = df['breast_birads'].apply(parse_birads)
df['density']    = df['breast_density'].apply(parse_density)

# Agregación a nivel mama (L o R): max BI-RADS de CC y MLO de esa mama, y moda de densidad
birads_breast = df.groupby(['study_id','laterality']).agg(
    birads  = ('birads_int', 'max'),
    density = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
).reset_index()
birads_L = birads_breast[birads_breast.laterality == 'L'].set_index('study_id')
birads_R = birads_breast[birads_breast.laterality == 'R'].set_index('study_id')

# Agregación a nivel estudio: max BI-RADS de las 4 vistas, moda de densidad, primer split
study_agg = df.groupby('study_id').agg(
    max_birads = ('birads_int', 'max'),
    density    = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
    split      = ('split', 'first'),
).reset_index()
study_agg['birads_L']  = study_agg['study_id'].map(birads_L['birads'])
study_agg['birads_R']  = study_agg['study_id'].map(birads_R['birads'])
study_agg['density_L'] = study_agg['study_id'].map(birads_L['density'])
study_agg['density_R'] = study_agg['study_id'].map(birads_R['density'])
study_agg['y_estudio'] = (study_agg['max_birads'] >= 4).astype(int)
study_agg['y_L']       = (study_agg['birads_L'] >= 4).astype(int)
study_agg['y_R']       = (study_agg['birads_R'] >= 4).astype(int)

print(f'Estudios: {len(study_agg):,}')
print(f'Positivos a nivel estudio: {study_agg.y_estudio.sum()} ({100*study_agg.y_estudio.mean():.2f}%)')
n_mamas = len(study_agg)*2; n_pos_mamas = study_agg.y_L.sum() + study_agg.y_R.sum()
print(f'Positivos a nivel mama:    {n_pos_mamas} de {n_mamas}  ({100*n_pos_mamas/n_mamas:.2f}%)')
print(f'Split: {study_agg.split.value_counts().to_dict()}')

# Índice study_id → {(L,CC): image_id, ...} para localizar archivos rápidamente
study_imgs = {}
for sid, grp in df.groupby('study_id'):
    study_imgs[sid] = {(r.laterality, r.view_position): r.image_id for _, r in grp.iterrows()}

Estudios: 5,000
Positivos a nivel estudio: 481 (9.62%)
Positivos a nivel mama:    494 de 10000  (4.94%)
Split: {'training': 4000, 'test': 1000}


## 4. Lógica de reanudación

Si ya hay archivos parciales, los cargamos y saltamos lo procesado. Esto permite parar el notebook con Ctrl+C y continuar después sin re-extraer.

In [5]:
OUT_VIEW = os.path.join(FEATURES_DIR, 'X_view.npy')
OUT_ASYM = os.path.join(FEATURES_DIR, 'X_asym.npy')
OUT_META = os.path.join(FEATURES_DIR, 'metadata.csv')
OUT_DONE = os.path.join(FEATURES_DIR, 'done_studies.txt')

done = set()
if os.path.isfile(OUT_DONE):
    with open(OUT_DONE) as f:
        done = set(l.strip() for l in f if l.strip())
    print(f'Reanudando: ya hechos {len(done)} estudios')

pendientes = [s for s in study_agg.study_id.tolist() if s not in done]
print(f'Pendientes: {len(pendientes)}')

Pendientes: 5000


## 5. Bucle principal

Por cada estudio:
1. Verifica que las 4 vistas existen en el CSV y en disco.
2. Ejecuta `extract_study()`.
3. Acumula en buffers en RAM.
4. Cada 100, vuelca a disco.

Muestra ETA cada 10 estudios para que sepas cuánto falta.

In [6]:
X_view_buf, X_asym_buf = [], []
meta_buf = []
SAVE_EVERY = 100

# Precargar buffers si hay parciales
if len(done) > 0:
    if os.path.isfile(OUT_VIEW):
        X_view_buf = list(np.load(OUT_VIEW))
        X_asym_buf = list(np.load(OUT_ASYM))
    if os.path.isfile(OUT_META):
        meta_buf = pd.read_csv(OUT_META).to_dict('records')
    print(f'Buffers precargados con {len(meta_buf)} filas')

def flush():
    """Vuelca los buffers a disco."""
    np.save(OUT_VIEW, np.array(X_view_buf, dtype=np.float32))
    np.save(OUT_ASYM, np.array(X_asym_buf, dtype=np.float32))
    pd.DataFrame(meta_buf).to_csv(OUT_META, index=False)
    with open(OUT_DONE, 'w') as f:
        for m in meta_buf:
            f.write(m['study_id'] + '\n')

t_global = time.time()
errores = []

for i, sid in enumerate(pendientes, start=1):
    row = study_agg[study_agg.study_id == sid].iloc[0]
    imgs = study_imgs.get(sid, {})
    required = [('L','CC'), ('R','CC'), ('L','MLO'), ('R','MLO')]
    if not all(k in imgs for k in required):
        errores.append((sid, 'faltan vistas en CSV')); continue
    vistas = {k: os.path.join(IMG_DIR, sid, f'{imgs[k]}.dicom') for k in required}
    if not all(os.path.isfile(p) for p in vistas.values()):
        errores.append((sid, 'DICOM ausente en disco')); continue

    t0 = time.time()
    result = extract_study(vistas)
    if result is None:
        errores.append((sid, 'fallo en extract')); continue
    X_view, X_asym = result

    X_view_buf.append(X_view)
    X_asym_buf.append(X_asym)
    meta_buf.append({
        'study_id'  : sid,
        'split'     : row['split'],
        'max_birads': int(row['max_birads']),
        'birads_L'  : int(row['birads_L']) if not pd.isna(row['birads_L']) else 1,
        'birads_R'  : int(row['birads_R']) if not pd.isna(row['birads_R']) else 1,
        'density'   : row['density'],
        'density_L' : row['density_L'],
        'density_R' : row['density_R'],
        'y_estudio' : int(row['y_estudio']),
        'y_L'       : int(row['y_L']),
        'y_R'       : int(row['y_R']),
    })
    done.add(sid)

    if i % 10 == 0 or i == 1:
        elapsed = time.time() - t_global
        rate = elapsed / i
        eta = rate * (len(pendientes) - i) / 60
        print(f'  [{i:5d}/{len(pendientes)}] sid={sid[:12]}...  '
              f'y_estudio={row["y_estudio"]}  {time.time()-t0:.1f}s/est  ETA {eta:.1f}min')

    if i % SAVE_EVERY == 0:
        flush()
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        print(f'    [flush · {len(meta_buf)} guardados]')

flush()
print(f'\nCompleto: {len(meta_buf)} estudios en {(time.time()-t_global)/60:.1f} min')
print(f'Errores: {len(errores)}')
if errores: print('Primeros 5:', errores[:5])

  [    1/5000] sid=0025a5dc99fd...  y_estudio=0  0.7s/est  ETA 57.9min
  [   10/5000] sid=008b8e61390f...  y_estudio=1  0.5s/est  ETA 45.5min
  [   20/5000] sid=00fe39c6d128...  y_estudio=0  0.5s/est  ETA 45.1min
  [   30/5000] sid=017ff0405a2e...  y_estudio=0  0.5s/est  ETA 43.4min
  [   40/5000] sid=01e354f1ba4b...  y_estudio=0  0.4s/est  ETA 42.0min
  [   50/5000] sid=0249157fa6d1...  y_estudio=0  0.5s/est  ETA 42.2min
  [   60/5000] sid=02f802e21fa4...  y_estudio=0  0.5s/est  ETA 42.1min
  [   70/5000] sid=039e97685515...  y_estudio=0  0.5s/est  ETA 42.2min
  [   80/5000] sid=041656eea2b3...  y_estudio=0  0.6s/est  ETA 42.0min
  [   90/5000] sid=0495f34b7e81...  y_estudio=0  0.6s/est  ETA 42.1min
  [  100/5000] sid=04fb5102b72c...  y_estudio=0  0.6s/est  ETA 41.8min
    [flush · 100 guardados]
  [  110/5000] sid=0575060d13f8...  y_estudio=0  0.6s/est  ETA 41.9min
  [  120/5000] sid=06394402b32d...  y_estudio=0  0.5s/est  ETA 42.0min
  [  130/5000] sid=0676f4386325...  y_estudio=0  

## 6. Verificación final

Comprobamos que los archivos guardados tienen los shapes esperados y la prevalencia coincide con lo planeado.

In [7]:
X_view = np.load(OUT_VIEW)
X_asym = np.load(OUT_ASYM)
meta   = pd.read_csv(OUT_META)

print(f'X_view: {X_view.shape}   {X_view.nbytes/1e6:.1f} MB    ← (N estudios, 4 vistas, 1024 dims pooleadas)')
print(f'X_asym: {X_asym.shape}   {X_asym.nbytes/1e6:.1f} MB    ← (N estudios, 2 pares, 1024 dims pooleadas)')
print(f'meta:   {len(meta)} filas')
print(f'\nPositivos a nivel estudio: {meta.y_estudio.sum()} ({100*meta.y_estudio.mean():.2f}%)')
print(f'Positivos a nivel mama (L+R): {(meta.y_L.sum()+meta.y_R.sum())} de {len(meta)*2}')
print(f'\nPrimeras filas del metadata:')
print(meta.head())

X_view: (4999, 4, 1024)   81.9 MB    ← (N estudios, 4 vistas, 1024 dims pooleadas)
X_asym: (4999, 2, 1024)   41.0 MB    ← (N estudios, 2 pares, 1024 dims pooleadas)
meta:   4999 filas

Positivos a nivel estudio: 481 (9.62%)
Positivos a nivel mama (L+R): 494 de 9998

Primeras filas del metadata:
                           study_id     split  max_birads  birads_L  birads_R  \
0  0025a5dc99fd5c742026f0b2b030d3e9      test           1         1         1   
1  0028fb2c7f0b3a5cb9a80cb0e1cdbb91  training           2         2         2   
2  0034765af074f93ed33d5e8399355caf  training           2         2         1   
3  003700f3c960e0b9bca2b8437c3dbf05  training           4         4         1   
4  004426a40da27ef22a866538b772ac44  training           1         1         1   

  density density_L density_R  y_estudio  y_L  y_R  
0       C         C         C          0    0    0  
1       C         C         C          0    0    0  
2       C         C         C          0    0    0  
3    

## Siguiente paso

`04_definicion_modelos.ipynb` — definimos las cabezas MLP y los wrappers de LightGBM para las distintas dimensionalidades (4096 para E-A, 2048 para E-B y M-A).